# Explore RNAseq Results with BigQuery Public Data

Your Nextflow pipeline produced gene expression quantification. Now let's query that data alongside **11,000 TCGA cancer samples** and **54 GTEx normal tissues** with zero data copy.

### Public Datasets Used
| Dataset | What | Console Link |
|---------|------|-------------|
| ISB-CGC TCGA | Gene expression from 33 cancer types | [Open in BigQuery](https://console.cloud.google.com/bigquery?p=isb-cgc-bq&d=TCGA&t=RNAseq_hg38_gdc_current&project=__PROJECT_ID__&ws=!1m5!1m4!4m3!1sisb-cgc-bq!2sTCGA!3sRNAseq_hg38_gdc_current) |
| ISB-CGC GTEx | Normal tissue expression (54 tissues) | [Open in BigQuery](https://console.cloud.google.com/bigquery?p=isb-cgc&d=GTEx_v7&t=gene_median_tpm&project=__PROJECT_ID__&ws=!1m5!1m4!4m3!1sisb-cgc!2sGTEx_v7!3sgene_median_tpm) |
| GENCODE | Gene annotation (symbols, types, coordinates) | [Open in BigQuery](https://console.cloud.google.com/bigquery?p=isb-cgc-bq&d=GENCODE&t=annotation_gtf_hg38_current&project=__PROJECT_ID__&ws=!1m5!1m4!4m3!1sisb-cgc-bq!2sGENCODE!3sannotation_gtf_hg38_current) |

Browse genomics datasets: [BigQuery Sharing](https://console.cloud.google.com/bigquery/analytics-hub) | [Cloud Marketplace](https://console.cloud.google.com/marketplace/browse?filter=solution-type:dataset&q=genomics)

In [ ]:
# Setup
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt

client = bigquery.Client()
print('BigQuery client ready')

## 1. Load Pipeline Results into BigQuery

Your nf-core/rnaseq pipeline produced gene-level expression data in `salmon.merged.gene_tpm.tsv`. Let's load it into BigQuery so we can JOIN with TCGA, GTEx, and GENCODE public datasets.

Columns: **gene_id** (Ensembl ENSG ID), **gene_name** (symbol), **TPM** (expression level per sample)

In [ ]:
import subprocess

# Find the gene-level TPM file from nf-core/rnaseq Salmon output
result = subprocess.run(
    ['gcloud', 'storage', 'ls', '-r', 'gs://__BUCKET_NAME__/'],
    capture_output=True, text=True
)
tpm_files = [line for line in result.stdout.split('\n') if line.endswith('salmon.merged.gene_tpm.tsv')]

if tpm_files:
    print(f'Found gene TPM file: {tpm_files[0]}')
    df_genes = pd.read_csv(tpm_files[0], sep='\t')

    # Rename columns for BigQuery JOINs: gene_id -> Name, sample column -> TPM
    sample_col = [c for c in df_genes.columns if c not in ('gene_id', 'gene_name')][0]
    df_load = df_genes.rename(columns={'gene_id': 'Name', sample_col: 'TPM'})
    print(f'Sample: {sample_col}')
    print(f'Loaded {len(df_load)} genes. Top 10 by expression:')
    display(df_load.sort_values('TPM', ascending=False).head(10))

    # Load into BigQuery (US multi-region, co-located with ISB-CGC public datasets)
    table_id = '__PROJECT_ID__.rnaseq_results.my_genes'
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')
    table_ref = bigquery.TableReference.from_string(table_id)
    job = client.load_table_from_dataframe(dataframe=df_load, destination=table_ref, job_config=job_config)
    job.result()
    print(f'\nLoaded {job.output_rows} rows into {table_id}')
else:
    print('No gene TPM file found. Run the pipeline first:')
    print('  nextflow run nf-core/rnaseq -r 3.19.0 --input samplesheet.csv \\')
    print('    --outdir gs://__BUCKET_NAME__/results --genome GRCh37 \\')
    print('    --pseudo_aligner salmon --skip_alignment')

## 2. Annotate Your Genes with GENCODE

JOIN your pipeline genes with [GENCODE](https://console.cloud.google.com/bigquery?p=isb-cgc-bq&d=GENCODE&t=annotation_gtf_hg38_current) to get gene symbols, types (protein-coding, lncRNA, etc.), and chromosomal locations.

In [ ]:
query_annotate = """
SELECT m.Name AS ensembl_id, ROUND(m.TPM, 2) AS my_tpm,
       g.gene_name, g.gene_type, g.seqname AS chromosome
FROM `__PROJECT_ID__.rnaseq_results.my_genes` m
JOIN `isb-cgc-bq.GENCODE.annotation_gtf_hg38_current` g
  ON m.Name = SPLIT(g.gene_id, ".")[OFFSET(0)]
WHERE g.feature = "gene"
ORDER BY m.TPM DESC
LIMIT 20
"""
df_annotated = client.query(query_annotate).to_dataframe()
print(f'Top 20 expressed genes from your pipeline (annotated via GENCODE):')
display(df_annotated)

## 3. Compare with TCGA Cancer Data

JOIN your top genes with [TCGA](https://console.cloud.google.com/bigquery?p=isb-cgc-bq&d=TCGA&t=RNAseq_hg38_gdc_current) — 11,000+ tumor samples across 33 cancer types. Are your highly-expressed genes also overexpressed in specific cancers?

In [ ]:
# Take top 5 protein-coding genes and compare with TCGA
query_tcga = """
WITH my_top_genes AS (
  SELECT m.Name, m.TPM
  FROM `__PROJECT_ID__.rnaseq_results.my_genes` m
  JOIN `isb-cgc-bq.GENCODE.annotation_gtf_hg38_current` g
    ON m.Name = SPLIT(g.gene_id, ".")[OFFSET(0)]
  WHERE g.feature = "gene" AND g.gene_type = "protein_coding"
  ORDER BY m.TPM DESC LIMIT 5
)
SELECT t.gene_name, ROUND(m.TPM, 2) AS my_tpm,
       t.project_short_name AS cancer_type,
       ROUND(AVG(t.tpm_unstranded), 2) AS tcga_avg_tpm,
       COUNT(DISTINCT t.case_barcode) AS n_patients
FROM my_top_genes m
JOIN `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current` t
  ON m.Name = t.Ensembl_gene_id
WHERE t.sample_type_name = "Primary Tumor"
GROUP BY 1, 2, 3
ORDER BY t.gene_name, tcga_avg_tpm DESC
"""
df_tcga = client.query(query_tcga).to_dataframe()

# Show first gene's cancer comparison
first_gene = df_tcga['gene_name'].iloc[0]
df_plot = df_tcga[df_tcga['gene_name'] == first_gene].head(10)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_plot['cancer_type'][::-1], df_plot['tcga_avg_tpm'][::-1], color='#4285F4')
ax.axvline(x=df_plot['my_tpm'].iloc[0], color='#EA4335', linestyle='--',
           label=f'My sample: {df_plot["my_tpm"].iloc[0]} TPM')
ax.set_xlabel('Average TPM')
ax.set_title(f'{first_gene}: My Sample vs TCGA Cancer Types')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Compare with GTEx Normal Tissue

JOIN your genes with [GTEx](https://console.cloud.google.com/bigquery?p=isb-cgc&d=GTEx_v7) — median expression across 54 normal human tissues. Is your expression abnormal compared to healthy tissue?

In [ ]:
# Compare your top expressed gene with normal tissue baselines
top_gene_id = df_annotated['ensembl_id'].iloc[0]
top_gene_name = df_annotated['gene_name'].iloc[0]
top_gene_tpm = df_annotated['my_tpm'].iloc[0]

query_gtex = f"""
SELECT g.gene_name, ROUND(t.gene_exp, 2) AS gtex_median_tpm,
       t.SMTSD AS normal_tissue
FROM `isb-cgc.GTEx_v7.gene_median_tpm` t
JOIN `isb-cgc-bq.GENCODE.annotation_gtf_hg38_current` g
  ON SPLIT(t.gene_id, ".")[OFFSET(0)] = SPLIT(g.gene_id, ".")[OFFSET(0)]
WHERE SPLIT(t.gene_id, ".")[OFFSET(0)] = "{top_gene_id}"
  AND g.feature = "gene"
ORDER BY gtex_median_tpm DESC LIMIT 15
"""
df_gtex = client.query(query_gtex).to_dataframe()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_gtex['normal_tissue'][::-1], df_gtex['gtex_median_tpm'][::-1], color='#4285F4')
ax.axvline(x=top_gene_tpm, color='#EA4335', linestyle='--',
           label=f'My sample: {top_gene_tpm} TPM')
ax.set_xlabel('Median TPM')
ax.set_title(f'{top_gene_name}: My Sample vs Normal Tissues (GTEx)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Which Cancer Type Does My Sample Resemble?

Compute expression correlation between your full gene profile and each TCGA cancer type's average profile across 20,000+ genes.

In [ ]:
query_similarity = """
WITH tcga_profile AS (
  SELECT t.project_short_name AS cancer_type, t.Ensembl_gene_id,
         AVG(t.tpm_unstranded) AS tcga_avg_tpm
  FROM `isb-cgc-bq.TCGA.RNAseq_hg38_gdc_current` t
  INNER JOIN `__PROJECT_ID__.rnaseq_results.my_genes` m
    ON t.Ensembl_gene_id = m.Name
  WHERE t.sample_type_name = "Primary Tumor" AND m.TPM > 0
  GROUP BY 1, 2
)
SELECT tp.cancer_type, COUNT(*) AS genes_compared,
       ROUND(CORR(m.TPM, tp.tcga_avg_tpm), 4) AS expression_correlation
FROM `__PROJECT_ID__.rnaseq_results.my_genes` m
JOIN tcga_profile tp ON m.Name = tp.Ensembl_gene_id
WHERE m.TPM > 0
GROUP BY 1 HAVING COUNT(*) >= 100
ORDER BY expression_correlation DESC LIMIT 10
"""
df_similarity = client.query(query_similarity).to_dataframe()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_similarity['cancer_type'][::-1],
        df_similarity['expression_correlation'][::-1], color='#34A853')
ax.set_xlabel('Pearson Correlation')
ax.set_title('Which TCGA Cancer Type Does My Sample Most Resemble?')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

top_match = df_similarity.iloc[0]
print(f'Top match: {top_match["cancer_type"]} (r={top_match["expression_correlation"]}, {top_match["genes_compared"]} genes compared)')

## 6. Ask Gemini to Summarize Findings

Use the Vertex AI Gemini SDK to get an AI-powered summary of the biological pathways implicated by the gene expression patterns above.

In [ ]:
# Install the Google Gen AI SDK (replaces deprecated vertexai.generative_models)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'google-genai'])

from google import genai
from google.genai.types import HttpOptions

# Initialize the Gen AI client for Vertex AI
client = genai.Client(
    http_options=HttpOptions(api_version='v1'),
    vertexai=True,
    project='__PROJECT_ID__',
    location='global'
)

# Build prompt from all analysis results
annotated_summary = df_annotated.to_string()
tcga_summary = df_tcga.head(20).to_string()
similarity_summary = df_similarity.to_string()

prompt = f"""You are a genomics expert. Based on the following RNAseq analysis results,
summarize the biological findings for a researcher.

Top expressed genes (annotated via GENCODE):
{annotated_summary}

Comparison with TCGA cancer types:
{tcga_summary}

Cancer type similarity (expression correlation):
{similarity_summary}

Please address:
1. What biological pathways do the top expressed genes participate in?
2. Why does the sample most resemble the top-matching cancer type?
3. What are potential therapeutic or research implications?"""

response = client.models.generate_content(
    model='gemini-3.1-pro-preview',
    contents=prompt
)
print(response.text)